# 03 — Generación de datos simulados
**Proyecto TechZone · Responsable: Hugo**

Este notebook genera datos sintéticos realistas para todas las tablas del modelo usando `Faker` y `pandas`.  
Los DataFrames resultantes quedan listos para ser cargados en BigQuery mediante la función `upload_dataframe` definida en `02_setup_bigquery.ipynb`.

**Tablas generadas:**
- `customers` — 500 clientes
- `categories` — 8 categorías de producto
- `products` — 70 productos
- `orders` — 2 000 pedidos
- `order_items` — ~4 500 líneas de pedido
- `payments` — 1 pago por pedido
- `reviews` — ~35 % de los productos entregados

## 1. Librerías

In [ ]:
import random
from datetime import timedelta

import pandas as pd
from faker import Faker

# Faker en español
fake = Faker('es_ES')
Faker.seed(42)
random.seed(42)

print("Librerías cargadas ✅")

## 4. Parámetros globales

In [ ]:
N_CUSTOMERS   = 500
N_PRODUCTS    = 70
N_ORDERS      = 2000
REVIEW_RATE   = 0.35   # % de order_items entregados que tienen review

# Países europeos donde opera TechZone
COUNTRIES = ["España", "Francia", "Alemania", "Italia", "Portugal", "Países Bajos"]

# Canales de adquisición
CHANNELS = ["organic", "paid_ads", "redes_sociales", "email", "referido"]

# Estados de pedido
ORDER_STATUSES = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]

# Métodos de pago
PAYMENT_METHODS = ["tarjeta_credito", "tarjeta_debito", "paypal", "transferencia", "bizum"]

# Estados de pago
PAYMENT_STATUSES = ["completed", "pending", "failed", "refunded"]

print("Parámetros configurados ✅")

## 5. Generación de tablas

### 5.1 `customers`

In [ ]:
customers = []

for i in range(1, N_CUSTOMERS + 1):
    country = random.choice(COUNTRIES)
    registered_at = fake.date_time_between(start_date="-3y", end_date="-1d")

    customers.append({
        "customer_id":          i,
        "first_name":           fake.first_name(),
        "last_name":            fake.last_name(),
        "email":                fake.unique.email(),
        "phone":                fake.phone_number(),
        "country":              country,
        "city":                 fake.city(),
        "acquisition_channel":  random.choice(CHANNELS),
        "registered_at":        registered_at
    })

df_customers = pd.DataFrame(customers)
print(f"customers: {len(df_customers)} filas")
df_customers.head(3)

### 5.2 `categories`

In [ ]:
# Categorías fijas — representativas de un e-commerce de tecnología
categories_data = [
    (1, "Smartphones",   "Teléfonos móviles de todas las gamas"),
    (2, "Laptops",       "Ordenadores portátiles para trabajo y gaming"),
    (3, "Audio",         "Auriculares, altavoces y accesorios de sonido"),
    (4, "Wearables",     "Smartwatches, pulseras de actividad y gafas inteligentes"),
    (5, "Periféricos",   "Teclados, ratones, monitores y accesorios de escritorio"),
    (6, "Tablets",       "Tabletas y accesorios"),
    (7, "Gaming",        "Consolas, mandos y accesorios de videojuegos"),
    (8, "Almacenamiento","Discos duros, SSDs, pendrives y tarjetas de memoria"),
]

df_categories = pd.DataFrame(categories_data, columns=["category_id", "name", "description"])
print(f"categories: {len(df_categories)} filas")
df_categories

### 5.3 `products`

In [ ]:
# Rangos de precio por categoría: (precio_venta_min, precio_venta_max, margen_aprox)
CATEGORY_PRICE_RANGES = {
    1: (199,  1200, 0.30),   # Smartphones
    2: (349,  2500, 0.25),   # Laptops
    3: (19,    400, 0.40),   # Audio
    4: (49,    600, 0.35),   # Wearables
    5: (9,     350, 0.38),   # Periféricos
    6: (129,   900, 0.28),   # Tablets
    7: (29,    700, 0.32),   # Gaming
    8: (9,     250, 0.42),   # Almacenamiento
}

# Prefijos de nombre por categoría para que sean reconocibles
PRODUCT_NAMES = {
    1: ["iPhone", "Samsung Galaxy", "Xiaomi", "OnePlus", "Google Pixel", "Motorola", "OPPO"],
    2: ["MacBook", "ThinkPad", "Dell XPS", "HP Spectre", "Asus ZenBook", "Acer Swift", "Lenovo IdeaPad"],
    3: ["Sony WH", "Bose QC", "AirPods", "JBL Tune", "Sennheiser HD", "Marshall Major", "Jabra Elite"],
    4: ["Apple Watch", "Fitbit", "Garmin", "Samsung Galaxy Watch", "Xiaomi Band", "Huawei Watch"],
    5: ["Logitech MX", "Razer DeathAdder", "Corsair K", "Samsung Monitor", "LG UltraWide", "Keychron K"],
    6: ["iPad Pro", "Samsung Tab", "Xiaomi Pad", "Lenovo Tab", "Huawei MatePad"],
    7: ["PS5 Controller", "Xbox Series", "Nintendo Switch", "Razer", "SteelSeries", "HyperX"],
    8: ["Samsung SSD", "WD My Passport", "Seagate", "Kingston", "SanDisk", "Crucial MX"],
}

products = []
product_id = 1
# Distribuir 70 productos entre 8 categorías
products_per_category = N_PRODUCTS // len(CATEGORY_PRICE_RANGES)
extra = N_PRODUCTS % len(CATEGORY_PRICE_RANGES)

for cat_id in range(1, 9):
    count = products_per_category + (1 if cat_id <= extra else 0)
    price_min, price_max, margin = CATEGORY_PRICE_RANGES[cat_id]
    names = PRODUCT_NAMES[cat_id]

    for j in range(count):
        sale_price = round(random.uniform(price_min, price_max), 2)
        cost_price = round(sale_price * (1 - margin), 2)
        brand = random.choice(names)
        suffix = fake.bothify(text="?##").upper()

        products.append({
            "product_id":   product_id,
            "category_id":  cat_id,
            "name":         f"{brand} {suffix}",
            "description":  fake.sentence(nb_words=8),
            "sale_price":   sale_price,
            "cost_price":   cost_price,
            "stock_qty":    random.randint(0, 300),
            "is_active":    random.choices([True, False], weights=[90, 10])[0]
        })
        product_id += 1

df_products = pd.DataFrame(products)
print(f"products: {len(df_products)} filas")
df_products.head(3)

### 5.4 `orders`

In [ ]:
# Precomputar registered_at por customer_id para garantizar registered_at < order_date
customer_reg = df_customers.set_index("customer_id")["registered_at"].to_dict()

orders = []

for order_id in range(1, N_ORDERS + 1):
    customer_id  = random.randint(1, N_CUSTOMERS)
    registered   = customer_reg[customer_id]

    # order_date siempre posterior a registered_at
    order_date   = fake.date_time_between(start_date=registered, end_date="now")

    status = random.choices(
        ORDER_STATUSES,
        weights=[5, 10, 15, 55, 10, 5]
    )[0]

    # Fechas de envío y entrega solo cuando tiene sentido
    shipped_at   = None
    delivered_at = None

    if status in ("shipped", "delivered", "returned"):
        shipped_at = order_date + timedelta(days=random.randint(1, 5))
    if status in ("delivered", "returned"):
        delivered_at = shipped_at + timedelta(days=random.randint(1, 10))

    country = random.choice(COUNTRIES)

    orders.append({
        "order_id":          order_id,
        "customer_id":       customer_id,
        "status":            status,
        "shipping_address":  fake.street_address(),
        "shipping_city":     fake.city(),
        "shipping_country":  country,
        "order_date":        order_date,
        "shipped_at":        shipped_at,
        "delivered_at":      delivered_at
    })

df_orders = pd.DataFrame(orders)
print(f"orders: {len(df_orders)} filas")
df_orders.head(3)

### 5.5 `order_items`

In [ ]:
# Precomputar precios por product_id
product_prices = df_products.set_index("product_id")["sale_price"].to_dict()
product_ids    = df_products["product_id"].tolist()

order_items = []
order_item_id = 1

for order_id in range(1, N_ORDERS + 1):
    # Entre 1 y 5 productos por pedido (media ~2.25 → total ~4500)
    n_items = random.choices([1, 2, 3, 4, 5], weights=[20, 35, 25, 15, 5])[0]
    # Sin repetir producto en el mismo pedido
    selected_products = random.sample(product_ids, k=min(n_items, len(product_ids)))

    for product_id in selected_products:
        sale_price      = product_prices[product_id]
        quantity        = random.randint(1, 4)
        # Pequeño descuento aleatorio (0–15 %)
        discount_pct    = random.choices([0, 0.05, 0.10, 0.15], weights=[50, 25, 15, 10])[0]
        discount_amount = round(sale_price * discount_pct, 2)

        order_items.append({
            "order_item_id":       order_item_id,
            "order_id":            order_id,
            "product_id":          product_id,
            "quantity":            quantity,
            "purchase_unit_price": sale_price,
            "discount_amount":     discount_amount
        })
        order_item_id += 1

df_order_items = pd.DataFrame(order_items)
print(f"order_items: {len(df_order_items)} filas")
df_order_items.head(3)

### 5.6 `payments`

In [ ]:
# Calcular importe total real de cada pedido desde order_items
order_totals = (
    df_order_items
    .assign(line_total=lambda df:
        (df["purchase_unit_price"] - df["discount_amount"]) * df["quantity"])
    .groupby("order_id")["line_total"]
    .sum()
    .round(2)
    .to_dict()
)

# Precomputar order_date por order_id
order_dates = df_orders.set_index("order_id")["order_date"].to_dict()
order_status_map = df_orders.set_index("order_id")["status"].to_dict()

payments = []

for payment_id, order_id in enumerate(range(1, N_ORDERS + 1), start=1):
    order_date    = order_dates[order_id]
    status        = order_status_map[order_id]
    amount        = order_totals.get(order_id, 0.0)

    # Estado de pago coherente con estado del pedido
    if status == "cancelled":
        payment_status = random.choices(["failed", "refunded"], weights=[60, 40])[0]
    elif status == "returned":
        payment_status = "refunded"
    elif status == "pending":
        payment_status = "pending"
    else:
        payment_status = "completed"

    # Fecha de pago: mismo día o hasta 2 días después del pedido
    payment_date = order_date + timedelta(hours=random.randint(0, 48))

    payments.append({
        "payment_id":     payment_id,
        "order_id":       order_id,
        "payment_method": random.choice(PAYMENT_METHODS),
        "payment_status": payment_status,
        "amount":         amount,
        "payment_date":   payment_date
    })

df_payments = pd.DataFrame(payments)
print(f"payments: {len(df_payments)} filas")
df_payments.head(3)

### 5.7 `reviews`

In [ ]:
# Solo se pueden valorar order_items de pedidos entregados
delivered_order_ids = set(
    df_orders.loc[df_orders["status"] == "delivered", "order_id"]
)

delivered_items = df_order_items[
    df_order_items["order_id"].isin(delivered_order_ids)
].copy()

# Muestra aleatoria del 35 %
review_sample = delivered_items.sample(frac=REVIEW_RATE, random_state=42)

# Precomputar delivered_at por order_id
delivered_at_map = df_orders.set_index("order_id")["delivered_at"].to_dict()

reviews = []

for review_id, (_, row) in enumerate(review_sample.iterrows(), start=1):
    delivered_at = delivered_at_map[row["order_id"]]
    # review_date: entre 1 y 30 días después de la entrega
    review_date  = delivered_at + timedelta(days=random.randint(1, 30))

    reviews.append({
        "review_id":     review_id,
        "order_item_id": int(row["order_item_id"]),
        "rating":        random.choices([1, 2, 3, 4, 5], weights=[5, 8, 15, 35, 37])[0],
        "comment":       fake.sentence(nb_words=random.randint(6, 18)),
        "review_date":   review_date
    })

df_reviews = pd.DataFrame(reviews)
print(f"reviews: {len(df_reviews)} filas")
df_reviews.head(3)

## 6. Resumen de DataFrames generados

In [ ]:
dfs = {
    "customers":   df_customers,
    "categories":  df_categories,
    "products":    df_products,
    "orders":      df_orders,
    "order_items": df_order_items,
    "payments":    df_payments,
    "reviews":     df_reviews,
}

for name, df in dfs.items():
    print(f"{name:15s}: {len(df):>6} filas | {df.shape[1]} columnas")

## 7. Validaciones de coherencia

Antes de cargar, comprobamos que las relaciones entre tablas son correctas.

In [ ]:
errores = []

# 1. orders.customer_id debe existir en customers
valid_customers = set(df_customers["customer_id"])
invalid = df_orders[~df_orders["customer_id"].isin(valid_customers)]
if not invalid.empty:
    errores.append(f"❌ orders con customer_id inválido: {len(invalid)}")

# 2. order_items.order_id debe existir en orders
valid_orders = set(df_orders["order_id"])
invalid = df_order_items[~df_order_items["order_id"].isin(valid_orders)]
if not invalid.empty:
    errores.append(f"❌ order_items con order_id inválido: {len(invalid)}")

# 3. order_items.product_id debe existir en products
valid_products = set(df_products["product_id"])
invalid = df_order_items[~df_order_items["product_id"].isin(valid_products)]
if not invalid.empty:
    errores.append(f"❌ order_items con product_id inválido: {len(invalid)}")

# 4. payments.order_id debe existir en orders
invalid = df_payments[~df_payments["order_id"].isin(valid_orders)]
if not invalid.empty:
    errores.append(f"❌ payments con order_id inválido: {len(invalid)}")

# 5. reviews.order_item_id debe existir en order_items
valid_items = set(df_order_items["order_item_id"])
invalid = df_reviews[~df_reviews["order_item_id"].isin(valid_items)]
if not invalid.empty:
    errores.append(f"❌ reviews con order_item_id inválido: {len(invalid)}")

# 6. Ratings entre 1 y 5
invalid = df_reviews[~df_reviews["rating"].between(1, 5)]
if not invalid.empty:
    errores.append(f"❌ reviews con rating fuera de rango: {len(invalid)}")

# 7. registered_at < order_date
merged = df_orders.merge(
    df_customers[["customer_id", "registered_at"]], on="customer_id"
)
invalid = merged[merged["registered_at"] >= merged["order_date"]]
if not invalid.empty:
    errores.append(f"❌ orders donde registered_at >= order_date: {len(invalid)}")

# 8. order_date < shipped_at < delivered_at (solo donde aplica)
with_ship = df_orders.dropna(subset=["shipped_at"])
invalid = with_ship[with_ship["order_date"] >= with_ship["shipped_at"]]
if not invalid.empty:
    errores.append(f"❌ orders donde order_date >= shipped_at: {len(invalid)}")

with_del = df_orders.dropna(subset=["shipped_at", "delivered_at"])
invalid = with_del[with_del["shipped_at"] >= with_del["delivered_at"]]
if not invalid.empty:
    errores.append(f"❌ orders donde shipped_at >= delivered_at: {len(invalid)}")

# Resultado
if errores:
    for e in errores:
        print(e)
else:
    print("✅ Todas las validaciones de coherencia pasadas correctamente")

## 8. Carga en BigQuery

⚠️ **Pendiente de credenciales.** Cuando el equipo tenga el `service_account.json`, ejecutar esta celda para cargar todos los datos.

In [ ]:
# DESCOMENTAR cuando se tengan las credenciales de Google Cloud

# import os
# from pathlib import Path
# from dotenv import load_dotenv
# from google.cloud import bigquery
#
# load_dotenv(dotenv_path='../.env')
# PROJECT_ID = os.getenv('GCP_PROJECT_ID')
# DATASET_ID = os.getenv('BQ_DATASET_ID')
# CREDENTIALS_PATH = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(Path('..') / CREDENTIALS_PATH)
# client = bigquery.Client(project=PROJECT_ID)
#
# def upload_dataframe(df, table_name):
#     table_id = f'{PROJECT_ID}.{DATASET_ID}.{table_name}'
#     job = client.load_table_from_dataframe(df, table_id)
#     job.result()
#     print(f'✅ {table_name}: {len(df)} filas cargadas')
#
# for table_name, df in dfs.items():
#     upload_dataframe(df, table_name)
#
# print('🎉 Carga completa')